In [1]:
import json
import matplotlib.pyplot as plt
import pandas as pd
import numpy as np

#### Data ingest and transform

In [2]:
# DATA INGEST: 

# Load the provided JSON file
file_path = 'DE_L_results_T_10_delta_5_scen_21_trial_1_inv_2_cap._1_cap.inc._5.json'
with open(file_path, 'r') as file:
    data = json.load(file)
    
# Create the combined data dictionary according to the provided mapping
combined_data = {
    "F_tender_schedule": data.get("F", {}),
    "Y_manufacturers_producing_period": data.get("Y", {}),
    "W_manufacturers_participate_tender": data.get("W", {}),
    "L_capacity_extension_decision": data.get("L", {}),
    "Q_commitment_amounts": data.get("Q", {}),
    "X_production_amounts": data.get("X", {}),
    "I_inventory_level": data.get("I", {}),
    "Vc_vaccinated_children": data.get("Vc", {}),
    "S_unvaccinated_children": data.get("S", {})
}

unique_producers = set(combined_data['Y_manufacturers_producing_period'].keys())
unique_vaccines = set(combined_data['Q_commitment_amounts'].keys())

# Load the scenario pair probabilities data from the provided file
scenario_probabilities_file_path = 'scenario_pair_probabilities_140.json'
with open(scenario_probabilities_file_path, 'r') as file:
    scenario_probabilities = json.load(file)
    
# Load the uploaded Excel file
file_path = '../../data/production_capacity_scenarios/production_capacity_scenarios.xlsx'
xls = pd.ExcelFile(file_path)

# Read the "master capacity" sheet
df_master_capacity = pd.read_excel(xls, sheet_name='master_capacity')
# df_master_capacity = df_master_capacity.sort_values(by='Manufacturer')
df_master_capacity.set_index(df_master_capacity.columns[0], inplace=False)

df_master_capacity, unique_producers, unique_vaccines

(       Manufacturer          1          2          3          4          5  \
 0       AJ_Vaccines    7711003    7711003    7711003    7711003    7711003   
 1          BB_NCIPD   39223956   39223956   39223956   39223956   39223956   
 2    Bharat_Biotech   61029105   61029105   61029105   61029105   61029105   
 3         Bilthoven   12048153   12048153   12048153   12048153   12048153   
 4      Biological_E  164885690  164885690  164885690  164885690  164885690   
 5    China_National   12812242   12812242   12812242   12812242   12812242   
 6               GSK  226762686  226762686  226762686  226762686  226762686   
 7      Haffkine_Bio   80972855   80972855   80972855   80972855   80972855   
 8           LG_Chem   43702188   43702188   43702188   43702188   43702188   
 9       Merck_Sharp   56991052   56991052   56991052   56991052   56991052   
 10           PT_Bio   45911731   45911731   45911731   45911731   45911731   
 11   Panacea_Biotec   10874165   10874165   1087416

#### L capacity increase

In [3]:
# Translate L capacity increase to useable data

# Adjusting the ordering function to handle nested dictionaries with non-numeric keys
def order_data(data_dict):
    ordered_data = {}
    for key, value in data_dict.items():
        if isinstance(value, dict):
            ordered_data[key] = {k: order_data(v) if isinstance(v, dict) else v for k, v in sorted(value.items(), key=lambda item: int(item[0]) if item[0].isdigit() else item[0])}
        else:
            ordered_data[key] = value
    return ordered_data

# Apply ordering to combined_data
L_ordered = order_data(combined_data['L_capacity_extension_decision'])

# Convert the 'L_capacity_extension_decision' data into a DataFrame for better visualization
L_df = pd.DataFrame(L_ordered).sort_index()
# Rotate the DataFrame
L_df_rotated = L_df.transpose()
# Order the columns from 1 to 10
ordered_columns = [str(i) for i in range(1, 11)]
L_df_rotated_ordered = L_df_rotated[ordered_columns]
# Reset the index to make manufacturers the first column
L_df_rotated_ordered.reset_index(inplace=True)
L_df_rotated_ordered.rename(columns={'index': 'Manufacturer'}, inplace=True)

L_df_rotated_ordered = L_df_rotated_ordered.sort_values(by='Manufacturer').reset_index(drop=True)
L_df_rotated_ordered
def transform_row(row):
    row = row.copy()
    if row.iloc[1] == 5.0:
        row.iloc[1] = 1.5
    else:
        row.iloc[1] = 1.0

    for i in range(2, len(row)):
        if row.iloc[i] == 5.0:
            row.iloc[i] = row.iloc[i-1] + 0.5
        else:
            row.iloc[i] = row.iloc[i-1]
    
    return row

# Apply the transformation to each row
transformed_capacity_increase = L_df_rotated_ordered.apply(transform_row, axis=1)
# Rename all columns except the first one to integers
transformed_capacity_increase.rename(columns={col: int(col) for col in transformed_capacity_increase.columns[1:]}, inplace=True)
transformed_capacity_increase.set_index(transformed_capacity_increase.columns[0], inplace=False)

# Performing the multiplication
adjusted_capacity = df_master_capacity.iloc[:, 1:11] * (transformed_capacity_increase.iloc[:, 1:11]/1)

# Adding the Manufacturer column back to the selected dataframe
adjusted_capacity['Manufacturer'] = df_master_capacity['Manufacturer']

adjusted_capacity = adjusted_capacity[[adjusted_capacity.columns[-1]] + list(adjusted_capacity.columns[:-1])]

adjusted_capacity.set_index(adjusted_capacity.columns[0], inplace=False)
adjusted_capacity

C:\Users\nicho\AppData\Local\Temp\ipykernel_22120\2984891616.py:25: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  L_df_rotated_ordered.rename(columns={'index': 'Manufacturer'}, inplace=True)


,Manufacturer,1,2,3,4,5,6,7,8,9,10
0,AJ_Vaccines,7711003.0,7.711003e+06,7.711003e+06,7.711003e+06,7.711003e+06,7.711003e+06,7.711003e+06,7.711003e+06,7.711003e+06,7.711003e+06
1,BB_NCIPD,39223956.0,3.922396e+07,3.922396e+07,3.922396e+07,3.922396e+07,3.922396e+07,3.922396e+07,3.922396e+07,3.922396e+07,3.922396e+07
2,Bharat_Biotech,61029105.0,6.102910e+07,6.102910e+07,6.102910e+07,6.102910e+07,6.102910e+07,6.102910e+07,6.102910e+07,6.102910e+07,6.102910e+07
3,Bilthoven,12048153.0,1.204815e+07,1.204815e+07,1.204815e+07,1.204815e+07,1.204815e+07,1.204815e+07,1.204815e+07,1.204815e+07,1.204815e+07
4,Biological_E,247328535.0,3.297714e+08,4.122142e+08,4.122142e+08,4.122142e+08,4.122142e+08,4.122142e+08,4.122142e+08,4.122142e+08,4.122142e+08
5,China_National,12812242.0,1.281224e+07,1.281224e+07,1.281224e+07,1.281224e+07,1.281224e+07,1.281224e+07,1.281224e+07,1.281224e+07,1.281224e+07
6,GSK,340144029.0,3.401440e+08,3.401440e+08,3.401440e+08,3.401440e+08,3.401440e+08,3.401440e+08,3.401440e+08,3.401440e+08,3.401440e+08
7,Haffkine_Bio,80972855.0,8.097286e+07,8.097286e+07,8.097286e+07,8.097286e+07,8.097286e+07,8.097286e+07,8.097286e+07,8.097286e+07,8.097286e+07
8,LG_Chem,43702188.0,4.370219e+07,4.370219e+07,4.370219e+07,4.370219e+07,4.370219e+07,4.370219e+07,4.370219e+07,4.370219e+07,4.370219e+07
9,Merck_Sharp,56991052.0,5.699105e+07,5.699105e+07,5.699105e+07,5.699105e+07,5.699105e+07,5.699105e+07,5.699105e+07,5.699105e+07,5.699105e+07


In [4]:
def find_fifth_level_keys_for_x(data):
    def recursive_search(d, level):
        if not isinstance(d, dict):
            return set()
        if level == 5:
            return set(d.keys())
        keys = set()
        for key, value in d.items():
            if isinstance(value, dict):
                keys.update(recursive_search(value, level + 1))
            elif level == 4 and key == 'X' and isinstance(value, dict):
                keys.update(value.keys())
        return keys

    return recursive_search(data, 1)

unique_keys_x_refined = find_fifth_level_keys_for_x(data)
# unique_keys_x_refined

# Extract the probabilities for the common scenarios
common_scenario_probabilities = {key: scenario_probabilities[key] for key in unique_keys_x_refined}

# Normalize the probabilities so their sum equals 1
total_probability = sum(common_scenario_probabilities.values())
normalized_scenario_probabilities = {key: value / total_probability for key, value in common_scenario_probabilities.items()}

normalized_scenario_probabilities
# Function to scale X production values using the normalized probabilities
def scale_x_production(data, vaccine_type, normalized_probabilities):
    if vaccine_type in data:
        # print(f"Vaccine: {vaccine_type}")
        for producer, years in data[vaccine_type].items():
            # print(f"Producer: {producer}")
            for year, scenarios in years.items():
                for scenario in scenarios:
                    if scenario in normalized_probabilities:
                        data[vaccine_type][producer][year][scenario] *= normalized_probabilities[scenario]
    return data

# Scale X production values for each producer and antigen
scaled_x_production_data = combined_data['X_production_amounts'].copy()
for vaccine_type in scaled_x_production_data:
    scaled_x_production_data = scale_x_production(scaled_x_production_data, vaccine_type, normalized_scenario_probabilities)


scaled_x_production_data


{'PCV': {'Serum_Institute': {'5': {'78': 1765.7280708341368,
    '79': 1795.9261580586967,
    '123': 777.1047704283308,
    '81': 70.85464774133149,
    '122': 341.3464577845524,
    '83': 28.548976469255592,
    '67': 2198.500041429992,
    '69': 2447.0928206108006,
    '68': 2447.0928206108006,
    '121': 847.1751360377094,
    '82': 23.731532138928216,
    '80': 2396.7354757721873,
    '125': 864.9749687248658,
    '84': 53.092932693235596,
    '64': 1765.7280708341368,
    '126': 864.9749687248658,
    '66': 2198.500041429992,
    '70': 1765.7280708341368,
    '65': 1795.9261580586967,
    '124': 283.74657992191925,
    '120': 51.23258071182087},
   '4': {'78': 341.7898802629391,
    '79': 282.6187383527849,
    '123': 0.0,
    '81': 11.104527992490745,
    '122': 0.0,
    '83': 0.0,
    '67': 0.0,
    '69': 473.6808435931723,
    '68': 473.6808435931723,
    '121': 132.77153034500557,
    '82': 0.0,
    '80': 377.1660395326632,
    '125': 166.898441175363,
    '84': 8.32086498330